In [1]:

import json
import pennylane as qml
import pennylane.numpy as np
from scipy.interpolate import interp1d

class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        num_wires = len(wire_order)
        mag = 0
        for i in range(num_wires):
            z_expval = np.sum(np.abs(state)**2 * (-1)**(np.arange(2**num_wires) >> i & 1))
            mag += z_expval
        abs_mag = np.abs(mag)
        return abs_mag / num_wires

def tfim_ground_state(num_qubits, h):
    wires = list(range(num_qubits))
    obs = [qml.PauliZ(wires[i]) @ qml.PauliZ(wires[(i + 1) % num_qubits]) for i in range(num_qubits)]
    obs += [qml.PauliX(wires[i]) for i in range(num_qubits)]
    coeffs = [-1 for _ in range(num_qubits)] + [-h for _ in range(num_qubits)]
    H = qml.Hamiltonian(coeffs, obs)
    H_mat = qml.matrix(H)
    eigvals, eigvecs = np.linalg.eigh(H_mat)
    return eigvecs[:, np.argmin(eigvals)]

dev = qml.device("default.qubit")

@qml.qnode(dev)
def magnetization(num_qubits, h):
    state = tfim_ground_state(num_qubits, h)
    qml.StatePrep(state, wires=list(range(num_qubits)))
    return AbsMagnetization(wires=list(range(num_qubits)))

def critical_point_estimate(mags, h_values):
    f = interp1d(h_values, mags, kind='cubic')
    h_interp = np.linspace(h_values[0], h_values[-1], 300)
    mags_interp = f(h_interp)
    differences = np.abs(np.diff(mags_interp))
    ind = np.argmax(differences)
    h_c = (h_interp[ind] + h_interp[ind + 1]) / 2
    return h_c

def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    if num_qubits <= 3:
        h_values = np.arange(0.01, 0.6, 0.005)
    elif num_qubits <= 5:
        h_values = np.arange(0.2, 1.1, 0.005)
    else:
        h_values = np.arange(0.5, 1.5, 0.005)
    mags = []
    for h in h_values:
        mags.append(magnetization(num_qubits, h))
    output = critical_point_estimate(np.array(mags), h_values)
    return str(output)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.isclose(solution_output, expected_output, rtol=5e-3)

test_cases = [
    ('5', '0.6735'),
    ('2', '0.3535')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")
    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except Exception as exc:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


Running test case 0 with input '5'...


Wrong Answer. Have: '0.21250000000000002'. Want: '0.6735'.
Running test case 1 with input '2'...


Wrong Answer. Have: '0.2775000000000001'. Want: '0.3535'.
